# Tests con respecto a los caps en la cantidad de columnas al importar viajes

In [1]:
import pandas as pd

from pylondrina.importing import import_trips_from_dataframe
from pylondrina.schema import FieldSpec, DomainSpec, TripSchema
from pylondrina.errors import ImportError as PylondrinaImportError

SOFT_CAP = 256
HARD_CAP = 1024

### Test 1 - Import normal de columnas

En esta prueba se construye un `TripSchema` pequeño y un `DataFrame` angosto, de modo que el import produzca un `TripDataset` normal. El objetivo es verificar que, en un caso sin ancho problemático, no aparezcan los codes nuevos de control de columnas: `IMP.COLUMNS.WIDE_TABLE` ni `IMP.COLUMNS.HARD_CAP_EXCEEDED`.

In [2]:
schema_normal = TripSchema(
    version="1.0-test",
    fields={
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "user_note": FieldSpec(name="user_note", dtype="string", required=True),
        "origin_time_utc": FieldSpec(name="origin_time_utc", dtype="datetime", required=False),
        "destination_time_utc": FieldSpec(name="destination_time_utc", dtype="datetime", required=False),
        "mode_hint": FieldSpec(name="mode_hint", dtype="string", required=False),
    },
    required=["movement_id", "user_note"],
)

df_normal = pd.DataFrame(
    {
        "movement_id": ["m0", "m1"],
        "user_note": ["fila_a", "fila_b"],
        "origin_time_utc": ["2024-01-01T08:00:00Z", "2024-01-01T09:00:00Z"],
        "destination_time_utc": ["2024-01-01T08:10:00Z", "2024-01-01T09:10:00Z"],
        "mode_hint": ["bus", "metro"],
    }
)

trips_normal, report_normal = import_trips_from_dataframe(
    df=df_normal,
    schema=schema_normal,
)

cap_issue_codes_normal = [
    issue.code
    for issue in report_normal.issues
    if issue.code.startswith("IMP.COLUMNS.")
]

print("Inspección de dataframe importado y de los Issues encontrados")
display(trips_normal.data)
display(report_normal.issues)

print("\nColumnas finales:", len(trips_normal.data.columns))
print("Codes de cap detectados:", cap_issue_codes_normal)
print("Todos los codes del reporte:", [issue.code for issue in report_normal.issues])

assert len(trips_normal.data.columns) < SOFT_CAP
assert "IMP.COLUMNS.WIDE_TABLE" not in cap_issue_codes_normal
assert "IMP.COLUMNS.HARD_CAP_EXCEEDED" not in cap_issue_codes_normal

print("OK: import normal sin warnings/errors de cap de columnas.")

Inspección de dataframe importado y de los Issues encontrados


,movement_id,user_note,origin_time_utc,destination_time_utc,mode_hint
0,m0,fila_a,2024-01-01 08:00:00+00:00,2024-01-01 08:10:00+00:00,bus
1,m1,fila_b,2024-01-01 09:00:00+00:00,2024-01-01 09:10:00+00:00,metro


[Issue(level='info', code='IMP.METADATA.DATASET_ID_CREATED', message="Se generó dataset_id para el dataset importado: 'tripds_82274e88ccea45f49821f3f064034c9e'.", field=None, source_field=None, row_count=None, details={'dataset_id': 'tripds_82274e88ccea45f49821f3f064034c9e', 'generator': 'uuid4_hex', 'stored_in': 'metadata.dataset_id'})]


Columnas finales: 5
Codes de cap detectados: []
Todos los codes del reporte: ['IMP.METADATA.DATASET_ID_CREATED']
OK: import normal sin warnings/errors de cap de columnas.


### Test 2 - Import con soft cap excedido usando solo campos del schema

En esta prueba se construye un `TripSchema` ancho, con más de 256 campos efectivos, pero sin sobrepasar el hard cap. El `DataFrame` contiene únicamente columnas que ya están declaradas en el schema, es decir, no se depende de campos extra o extendidos. El objetivo es verificar que el import continúe, pero emita el warning `IMP.COLUMNS.WIDE_TABLE` y no el error de hard cap.

In [3]:
n_wide_schema_fields = 260

fields_soft_schema = {
    "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
    "user_note": FieldSpec(name="user_note", dtype="string", required=True),
    "origin_time_utc": FieldSpec(name="origin_time_utc", dtype="datetime", required=False),
    "destination_time_utc": FieldSpec(name="destination_time_utc", dtype="datetime", required=False),
}

for i in range(n_wide_schema_fields):
    field_name = f"wide_{i:03d}"
    fields_soft_schema[field_name] = FieldSpec(
        name=field_name,
        dtype="string",
        required=False,
    )

schema_soft_schema_only = TripSchema(
    version="1.0-test",
    fields=fields_soft_schema,
    required=["movement_id", "user_note"],
)

row_soft_schema = {
    "movement_id": "m0",
    "user_note": "ok",
    "origin_time_utc": "2024-01-01T08:00:00Z",
    "destination_time_utc": "2024-01-01T08:10:00Z",
}

for i in range(n_wide_schema_fields):
    row_soft_schema[f"wide_{i:03d}"] = f"valor_{i}"

df_soft_schema = pd.DataFrame([row_soft_schema])

trips_soft_schema, report_soft_schema = import_trips_from_dataframe(
    df=df_soft_schema,
    schema=schema_soft_schema_only,
)

cap_issue_codes_soft_schema = [
    issue.code
    for issue in report_soft_schema.issues
    if issue.code.startswith("IMP.COLUMNS.")
]

wide_issue = next(
    issue
    for issue in report_soft_schema.issues
    if issue.code == "IMP.COLUMNS.WIDE_TABLE"
)

print("Inspección de dataframe importado y de los Issues encontrados")
display(trips_soft_schema.data)
display(report_soft_schema.issues)

print("Columnas finales:", len(trips_soft_schema.data.columns))
print("Codes de cap detectados:", cap_issue_codes_soft_schema)
print("Detalles del warning:", wide_issue.details)
print("extra_fields_kept:", trips_soft_schema.metadata.get("extra_fields_kept"))

assert SOFT_CAP < len(trips_soft_schema.data.columns) < HARD_CAP
assert "IMP.COLUMNS.WIDE_TABLE" in cap_issue_codes_soft_schema
assert "IMP.COLUMNS.HARD_CAP_EXCEEDED" not in cap_issue_codes_soft_schema

# Este caso debe ser schema-only, sin columnas extra conservadas.
assert trips_soft_schema.metadata.get("extra_fields_kept") == []

print("OK: soft cap detectado correctamente con columnas definidas solo en el schema.")

Inspección de dataframe importado y de los Issues encontrados


,movement_id,user_note,origin_time_utc,destination_time_utc,wide_000,wide_001,wide_002,wide_003,wide_004,wide_005,...,wide_250,wide_251,wide_252,wide_253,wide_254,wide_255,wide_256,wide_257,wide_258,wide_259
0,m0,ok,2024-01-01 08:00:00+00:00,2024-01-01 08:10:00+00:00,valor_0,valor_1,valor_2,valor_3,valor_4,valor_5,...,valor_250,valor_251,valor_252,valor_253,valor_254,valor_255,valor_256,valor_257,valor_258,valor_259


[Issue(level='warning', code='IMP.COLUMNS.WIDE_TABLE', message='El TripDataset resultante tiene 264 columnas, superando el soft cap 256; se permite continuar, pero la tabla es ancha y puede generar fricción operativa.', field=None, source_field=None, row_count=None, details={'n_columns': 264, 'soft_cap': 256, 'hard_cap': 1024, 'extra_fields_kept_sample': [], 'extra_fields_kept_total': 0, 'action': 'allow_with_warning'}),
 Issue(level='info', code='IMP.METADATA.DATASET_ID_CREATED', message="Se generó dataset_id para el dataset importado: 'tripds_9e65124b55e64a75aa59ce9a5d65a549'.", field=None, source_field=None, row_count=None, details={'dataset_id': 'tripds_9e65124b55e64a75aa59ce9a5d65a549', 'generator': 'uuid4_hex', 'stored_in': 'metadata.dataset_id'})]

Columnas finales: 264
Codes de cap detectados: ['IMP.COLUMNS.WIDE_TABLE']
Detalles del warning: {'n_columns': 264, 'soft_cap': 256, 'hard_cap': 1024, 'extra_fields_kept_sample': [], 'extra_fields_kept_total': 0, 'action': 'allow_with_warning'}
extra_fields_kept: []
OK: soft cap detectado correctamente con columnas definidas solo en el schema.


### Test 3 - Import con hard cap excedido usando solo campos extra

En esta prueba se construye un `TripSchema` pequeño y se fuerza el exceso de columnas mediante campos extra que no forman parte del schema. El objetivo es verificar que el import sea rechazado con la excepción correspondiente y que el code/issue gatillante sea `IMP.COLUMNS.HARD_CAP_EXCEEDED`.

In [4]:
n_extra_fields_hard = 1022

schema_hard_extras_only = TripSchema(
    version="1.0-test",
    fields={
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "core_col": FieldSpec(name="core_col", dtype="string", required=True),
        "origin_time_utc": FieldSpec(name="origin_time_utc", dtype="datetime", required=False),
        "destination_time_utc": FieldSpec(name="destination_time_utc", dtype="datetime", required=False),
    },
    required=["movement_id", "core_col"],
)

row_hard_extras = {
    "movement_id": "m0",
    "core_col": "ok",
    "origin_time_utc": "2024-01-01T08:00:00Z",
    "destination_time_utc": "2024-01-01T08:10:00Z",
}

for i in range(n_extra_fields_hard):
    row_hard_extras[f"extra_{i:04d}"] = f"valor_{i}"

df_hard_extras = pd.DataFrame([row_hard_extras])

print("Inspección de dataframe WIDE que se quiere importar")
display(df_hard_extras)

try:
    _trips_hard, _report_hard = import_trips_from_dataframe(
        df=df_hard_extras,
        schema=schema_hard_extras_only,
    )
    raise AssertionError("Se esperaba una excepción por hard cap, pero el import continuó.")
except PylondrinaImportError as exc:
    print("Tipo de excepción:", type(exc).__name__)
    print("Code de la excepción:", exc.code)
    print("Code del issue gatillante:", exc.issue.code if exc.issue is not None else None)
    print("Details:", exc.details)
    print("Issues acumulados:", [issue.code for issue in (exc.issues or [])])

    assert exc.code == "IMP.COLUMNS.HARD_CAP_EXCEEDED"
    assert exc.issue is not None
    assert exc.issue.code == "IMP.COLUMNS.HARD_CAP_EXCEEDED"
    assert exc.details["n_columns"] > HARD_CAP
    assert exc.details["extra_fields_kept_total"] == n_extra_fields_hard
    assert [issue.code for issue in (exc.issues or [])] == ["IMP.COLUMNS.HARD_CAP_EXCEEDED"]

    print("OK: hard cap detectado correctamente con columnas extra fuera del schema.")

Inspección de dataframe WIDE que se quiere importar


,movement_id,core_col,origin_time_utc,destination_time_utc,extra_0000,extra_0001,extra_0002,extra_0003,extra_0004,extra_0005,...,extra_1012,extra_1013,extra_1014,extra_1015,extra_1016,extra_1017,extra_1018,extra_1019,extra_1020,extra_1021
0,m0,ok,2024-01-01T08:00:00Z,2024-01-01T08:10:00Z,valor_0,valor_1,valor_2,valor_3,valor_4,valor_5,...,valor_1012,valor_1013,valor_1014,valor_1015,valor_1016,valor_1017,valor_1018,valor_1019,valor_1020,valor_1021


Tipo de excepción: ImportError
Code de la excepción: IMP.COLUMNS.HARD_CAP_EXCEEDED
Code del issue gatillante: IMP.COLUMNS.HARD_CAP_EXCEEDED
Details: {'n_columns': 1026, 'soft_cap': 256, 'hard_cap': 1024, 'extra_fields_kept_sample': ['extra_0000', 'extra_0001', 'extra_0002', 'extra_0003', 'extra_0004', 'extra_0005', 'extra_0006', 'extra_0007', 'extra_0008', 'extra_0009'], 'extra_fields_kept_total': 1022, 'action': 'abort'}
Issues acumulados: ['IMP.COLUMNS.HARD_CAP_EXCEEDED']
OK: hard cap detectado correctamente con columnas extra fuera del schema.


# Tests con respecto a la politica de inferencia de categorias al importar viajes

### Test 1 - Política de inferencia categórica con un dataset de tamaño normal

En esta prueba se construye un `TripDataset` con 200 filas, es decir, un tamaño normal dentro del rango esperado para una verificación rápida. Se definen dos campos categóricos con `DomainSpec.values=[]`: uno cuya cardinalidad sí cumple la política de inferencia y otro cuya cardinalidad la excede. Además, se incluye un tercer campo categórico con dominio explícito para comprobar que el comportamiento previo de los dominios definidos no fue alterado por este cambio.

El objetivo es verificar que el import:
- emita el code informativo de inferencia aplicada para el campo que sí cumple la política,
- emita el warning de degradación a `string` para el campo que no cumple la política,
- mantenga sin cambios el campo categórico con dominio explícito,
- y refleje correctamente esos resultados en `schema_effective` y `domains_effective`.

In [5]:
n_rows = 200
cardinality_limit = 0.05 * n_rows  # 10.0

schema_normal_rows = TripSchema(
    version="1.0-test",
    fields={
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "origin_time_utc": FieldSpec(name="origin_time_utc", dtype="datetime", required=False),
        "destination_time_utc": FieldSpec(name="destination_time_utc", dtype="datetime", required=False),

        # Campo categórico declarado por el usuario, pero sin dominio explícito.
        # Debe pasar la política: k = 10 <= 10.0
        "cat_infer_ok": FieldSpec(
            name="cat_infer_ok",
            dtype="categorical",
            required=False,
            domain=DomainSpec(values=[], extendable=True),
        ),

        # Campo categórico declarado por el usuario, pero sin dominio explícito.
        # Debe fallar la política: k = 11 > 10.0
        "cat_infer_bad": FieldSpec(
            name="cat_infer_bad",
            dtype="categorical",
            required=False,
            domain=DomainSpec(values=[], extendable=True),
        ),

        # Campo categórico normal con dominio explícito.
        "cat_defined": FieldSpec(
            name="cat_defined",
            dtype="categorical",
            required=False,
            domain=DomainSpec(values=["A", "B", "C"], extendable=False),
        ),
    },
    required=["movement_id", "user_id"],
)

df_normal_rows = pd.DataFrame(
    {
        "movement_id": [f"m{i}" for i in range(n_rows)],
        "user_id": [f"user_{i}" for i in range(n_rows)],
        "origin_time_utc": ["2024-01-01T08:00:00Z"] * n_rows,
        "destination_time_utc": ["2024-01-01T08:10:00Z"] * n_rows,

        # 10 valores únicos
        "cat_infer_ok": [f"ok_{i % 10:02d}" for i in range(n_rows)],

        # 11 valores únicos
        "cat_infer_bad": [f"bad_{i % 11:02d}" for i in range(n_rows)],

        # Dominio explícito
        "cat_defined": [["A", "B", "C"][i % 3] for i in range(n_rows)],
    }
)

trips_normal_rows, report_normal_rows = import_trips_from_dataframe(
    df=df_normal_rows,
    schema=schema_normal_rows,
)

issue_codes_set = {issue.code for issue in report_normal_rows.issues}
empty_domain_fields = {
    issue.field
    for issue in report_normal_rows.issues
    if issue.code == "SCH.DOMAIN.EMPTY_VALUES"
}

applied_issues = [
    issue
    for issue in report_normal_rows.issues
    if issue.code == "DOM.INFERENCE.APPLIED" and issue.field == "cat_infer_ok"
]
degraded_issues = [
    issue
    for issue in report_normal_rows.issues
    if issue.code == "DOM.INFERENCE.DEGRADED_TO_STRING" and issue.field == "cat_infer_bad"
]
defined_field_inference_issues = [
    issue
    for issue in report_normal_rows.issues
    if issue.field == "cat_defined" and issue.code.startswith("DOM.INFERENCE.")
]

print("Inspección de dataframe importado y de los Issues encontrados")
display(trips_normal_rows.data)
display(report_normal_rows.issues)

print("Codes detectados:", sorted(issue_codes_set))
print("Campos con SCH.DOMAIN.EMPTY_VALUES:", sorted(empty_domain_fields))
print("dtype_effective:", trips_normal_rows.schema_effective.dtype_effective)
print("domains_effective keys:", sorted(trips_normal_rows.metadata["domains_effective"].keys()))

assert {"cat_infer_ok", "cat_infer_bad"}.issubset(empty_domain_fields)

assert "DOM.INFERENCE.APPLIED" in issue_codes_set
assert "DOM.INFERENCE.DEGRADED_TO_STRING" in issue_codes_set

assert applied_issues, "No se encontró el issue DOM.INFERENCE.APPLIED para cat_infer_ok."
assert degraded_issues, "No se encontró el issue DOM.INFERENCE.DEGRADED_TO_STRING para cat_infer_bad."

applied_issue = applied_issues[0]
degraded_issue = degraded_issues[0]

assert applied_issue.details["alpha"] == 0.05
assert applied_issue.details["n_rows_non_null"] == n_rows
assert applied_issue.details["n_unique_observed"] == 10
assert applied_issue.details["cardinality_limit"] == cardinality_limit

assert degraded_issue.details["alpha"] == 0.05
assert degraded_issue.details["n_rows_non_null"] == n_rows
assert degraded_issue.details["n_unique_observed"] == 11
assert degraded_issue.details["cardinality_limit"] == cardinality_limit
assert degraded_issue.details["fallback_dtype"] == "string"

# Campo que sí cumple política: sigue categórico y queda en domains_effective.
assert trips_normal_rows.schema_effective.dtype_effective["cat_infer_ok"] == "categorical"
assert "cat_infer_ok" in trips_normal_rows.metadata["domains_effective"]
assert len(trips_normal_rows.metadata["domains_effective"]["cat_infer_ok"]["values"]) == 10

# Campo que no cumple política: se degrada a string y desaparece de domains_effective.
assert trips_normal_rows.schema_effective.dtype_effective["cat_infer_bad"] == "string"
assert "cat_infer_bad" not in trips_normal_rows.metadata["domains_effective"]

# Campo con dominio explícito: se mantiene categórico y no entra a la política de inferencia.
assert trips_normal_rows.schema_effective.dtype_effective["cat_defined"] == "categorical"
assert "cat_defined" in trips_normal_rows.metadata["domains_effective"]
assert trips_normal_rows.metadata["domains_effective"]["cat_defined"]["base_values"] == ["A", "B", "C"]
assert not defined_field_inference_issues

print("OK: en el caso de tamaño normal se verificó inferencia aplicada, degradación a string y preservación del campo con dominio explícito.")

Inspección de dataframe importado y de los Issues encontrados


,movement_id,user_id,origin_time_utc,destination_time_utc,cat_infer_ok,cat_infer_bad,cat_defined
0,m0,user_0,2024-01-01 08:00:00+00:00,2024-01-01 08:10:00+00:00,ok_00,bad_00,A
1,m1,user_1,2024-01-01 08:00:00+00:00,2024-01-01 08:10:00+00:00,ok_01,bad_01,B
2,m2,user_2,2024-01-01 08:00:00+00:00,2024-01-01 08:10:00+00:00,ok_02,bad_02,C
3,m3,user_3,2024-01-01 08:00:00+00:00,2024-01-01 08:10:00+00:00,ok_03,bad_03,A
4,m4,user_4,2024-01-01 08:00:00+00:00,2024-01-01 08:10:00+00:00,ok_04,bad_04,B
...,...,...,...,...,...,...,...
195,m195,user_195,2024-01-01 08:00:00+00:00,2024-01-01 08:10:00+00:00,ok_05,bad_08,A
196,m196,user_196,2024-01-01 08:00:00+00:00,2024-01-01 08:10:00+00:00,ok_06,bad_09,B
197,m197,user_197,2024-01-01 08:00:00+00:00,2024-01-01 08:10:00+00:00,ok_07,bad_10,C
198,m198,user_198,2024-01-01 08:00:00+00:00,2024-01-01 08:10:00+00:00,ok_08,bad_00,A


[Issue(level='info', code='SCH.DOMAIN.EMPTY_VALUES', message="El DomainSpec del campo 'cat_infer_ok' tiene values vacío; se tratará como dominio bootstrap/extensible según política.", field='cat_infer_ok', source_field=None, row_count=None, details={'field': 'cat_infer_ok', 'values_size': 0, 'extendable': True, 'note': 'candidate_bootstrap'}),
 Issue(level='info', code='SCH.DOMAIN.EMPTY_VALUES', message="El DomainSpec del campo 'cat_infer_bad' tiene values vacío; se tratará como dominio bootstrap/extensible según política.", field='cat_infer_bad', source_field=None, row_count=None, details={'field': 'cat_infer_bad', 'values_size': 0, 'extendable': True, 'note': 'candidate_bootstrap'}),
 Issue(level='info', code='DOM.INFERENCE.APPLIED', message="Se infirió el dominio efectivo de 'cat_infer_ok' a partir de los valores observados (10 únicos sobre 200 filas no nulas); el campo se mantiene como categórico.", field='cat_infer_ok', source_field=None, row_count=None, details={'field': 'cat_inf

Codes detectados: ['DOM.INFERENCE.APPLIED', 'DOM.INFERENCE.DEGRADED_TO_STRING', 'IMP.METADATA.DATASET_ID_CREATED', 'SCH.DOMAIN.EMPTY_VALUES']
Campos con SCH.DOMAIN.EMPTY_VALUES: ['cat_infer_bad', 'cat_infer_ok']
dtype_effective: {'movement_id': 'string', 'user_id': 'string', 'origin_time_utc': 'datetime', 'destination_time_utc': 'datetime', 'cat_infer_ok': 'categorical', 'cat_infer_bad': 'string', 'cat_defined': 'categorical'}
domains_effective keys: ['cat_defined', 'cat_infer_ok']
OK: en el caso de tamaño normal se verificó inferencia aplicada, degradación a string y preservación del campo con dominio explícito.


### Test 2 - Política de inferencia categórica con un dataset de mayor tamaño

En esta prueba se repite la misma lógica del caso anterior, pero ahora con 20.000 filas, de modo que la cardinalidad permitida por la política aumente. El objetivo es comprobar que el comportamiento siga siendo consistente cuando cambia la escala del dataset: un campo debe continuar siendo aceptado como categórico por cumplir la política, otro debe degradarse a `string` por excederla, y el campo con dominio explícito debe permanecer estable.

Con 20.000 filas, el límite queda en `0.05 * 20000 = 1000`. Por ello, se construye un campo con exactamente 1000 valores únicos y otro con 1001.

In [6]:
n_rows = 20_000
cardinality_limit = 0.05 * n_rows  # 1000.0

schema_large_rows = TripSchema(
    version="1.0-test",
    fields={
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "origin_time_utc": FieldSpec(name="origin_time_utc", dtype="datetime", required=False),
        "destination_time_utc": FieldSpec(name="destination_time_utc", dtype="datetime", required=False),

        "cat_infer_ok": FieldSpec(
            name="cat_infer_ok",
            dtype="categorical",
            required=False,
            domain=DomainSpec(values=[], extendable=True),
        ),
        "cat_infer_bad": FieldSpec(
            name="cat_infer_bad",
            dtype="categorical",
            required=False,
            domain=DomainSpec(values=[], extendable=True),
        ),
        "cat_defined": FieldSpec(
            name="cat_defined",
            dtype="categorical",
            required=False,
            domain=DomainSpec(values=["A", "B", "C"], extendable=False),
        ),
    },
    required=["movement_id", "user_id"],
)

df_large_rows = pd.DataFrame(
    {
        "movement_id": [f"m{i}" for i in range(n_rows)],
        "user_id": [f"user_{i}" for i in range(n_rows)],
        "origin_time_utc": ["2024-01-01T08:00:00Z"] * n_rows,
        "destination_time_utc": ["2024-01-01T08:10:00Z"] * n_rows,

        # 1000 valores únicos
        "cat_infer_ok": [f"ok_{i % 1000:04d}" for i in range(n_rows)],

        # 1001 valores únicos
        "cat_infer_bad": [f"bad_{i % 1001:04d}" for i in range(n_rows)],

        "cat_defined": [["A", "B", "C"][i % 3] for i in range(n_rows)],
    }
)

trips_large_rows, report_large_rows = import_trips_from_dataframe(
    df=df_large_rows,
    schema=schema_large_rows,
)

issue_codes_set = {issue.code for issue in report_large_rows.issues}
empty_domain_fields = {
    issue.field
    for issue in report_large_rows.issues
    if issue.code == "SCH.DOMAIN.EMPTY_VALUES"
}

applied_issues = [
    issue
    for issue in report_large_rows.issues
    if issue.code == "DOM.INFERENCE.APPLIED" and issue.field == "cat_infer_ok"
]
degraded_issues = [
    issue
    for issue in report_large_rows.issues
    if issue.code == "DOM.INFERENCE.DEGRADED_TO_STRING" and issue.field == "cat_infer_bad"
]
defined_field_inference_issues = [
    issue
    for issue in report_large_rows.issues
    if issue.field == "cat_defined" and issue.code.startswith("DOM.INFERENCE.")
]

print("Inspección de dataframe importado y de los Issues encontrados")
display(trips_large_rows.data)
display(report_large_rows.issues)

print("\nCodes detectados:", sorted(issue_codes_set))
print("Campos con SCH.DOMAIN.EMPTY_VALUES:", sorted(empty_domain_fields))
print("dtype_effective:", trips_large_rows.schema_effective.dtype_effective)
print("domains_effective keys:", sorted(trips_large_rows.metadata["domains_effective"].keys()))

assert {"cat_infer_ok", "cat_infer_bad"}.issubset(empty_domain_fields)

assert "DOM.INFERENCE.APPLIED" in issue_codes_set
assert "DOM.INFERENCE.DEGRADED_TO_STRING" in issue_codes_set

assert applied_issues, "No se encontró el issue DOM.INFERENCE.APPLIED para cat_infer_ok."
assert degraded_issues, "No se encontró el issue DOM.INFERENCE.DEGRADED_TO_STRING para cat_infer_bad."

applied_issue = applied_issues[0]
degraded_issue = degraded_issues[0]

assert applied_issue.details["alpha"] == 0.05
assert applied_issue.details["n_rows_non_null"] == n_rows
assert applied_issue.details["n_unique_observed"] == 1000
assert applied_issue.details["cardinality_limit"] == cardinality_limit

assert degraded_issue.details["alpha"] == 0.05
assert degraded_issue.details["n_rows_non_null"] == n_rows
assert degraded_issue.details["n_unique_observed"] == 1001
assert degraded_issue.details["cardinality_limit"] == cardinality_limit
assert degraded_issue.details["fallback_dtype"] == "string"

assert trips_large_rows.schema_effective.dtype_effective["cat_infer_ok"] == "categorical"
assert "cat_infer_ok" in trips_large_rows.metadata["domains_effective"]
assert len(trips_large_rows.metadata["domains_effective"]["cat_infer_ok"]["values"]) == 1000

assert trips_large_rows.schema_effective.dtype_effective["cat_infer_bad"] == "string"
assert "cat_infer_bad" not in trips_large_rows.metadata["domains_effective"]

assert trips_large_rows.schema_effective.dtype_effective["cat_defined"] == "categorical"
assert "cat_defined" in trips_large_rows.metadata["domains_effective"]
assert trips_large_rows.metadata["domains_effective"]["cat_defined"]["base_values"] == ["A", "B", "C"]
assert not defined_field_inference_issues

print("OK: en el caso de mayor tamaño se verificó que la política sigue comportándose igual al cambiar la escala del dataset.")

Inspección de dataframe importado y de los Issues encontrados


,movement_id,user_id,origin_time_utc,destination_time_utc,cat_infer_ok,cat_infer_bad,cat_defined
0,m0,user_0,2024-01-01 08:00:00+00:00,2024-01-01 08:10:00+00:00,ok_0000,bad_0000,A
1,m1,user_1,2024-01-01 08:00:00+00:00,2024-01-01 08:10:00+00:00,ok_0001,bad_0001,B
2,m2,user_2,2024-01-01 08:00:00+00:00,2024-01-01 08:10:00+00:00,ok_0002,bad_0002,C
3,m3,user_3,2024-01-01 08:00:00+00:00,2024-01-01 08:10:00+00:00,ok_0003,bad_0003,A
4,m4,user_4,2024-01-01 08:00:00+00:00,2024-01-01 08:10:00+00:00,ok_0004,bad_0004,B
...,...,...,...,...,...,...,...
19995,m19995,user_19995,2024-01-01 08:00:00+00:00,2024-01-01 08:10:00+00:00,ok_0995,bad_0976,A
19996,m19996,user_19996,2024-01-01 08:00:00+00:00,2024-01-01 08:10:00+00:00,ok_0996,bad_0977,B
19997,m19997,user_19997,2024-01-01 08:00:00+00:00,2024-01-01 08:10:00+00:00,ok_0997,bad_0978,C
19998,m19998,user_19998,2024-01-01 08:00:00+00:00,2024-01-01 08:10:00+00:00,ok_0998,bad_0979,A


[Issue(level='info', code='SCH.DOMAIN.EMPTY_VALUES', message="El DomainSpec del campo 'cat_infer_ok' tiene values vacío; se tratará como dominio bootstrap/extensible según política.", field='cat_infer_ok', source_field=None, row_count=None, details={'field': 'cat_infer_ok', 'values_size': 0, 'extendable': True, 'note': 'candidate_bootstrap'}),
 Issue(level='info', code='SCH.DOMAIN.EMPTY_VALUES', message="El DomainSpec del campo 'cat_infer_bad' tiene values vacío; se tratará como dominio bootstrap/extensible según política.", field='cat_infer_bad', source_field=None, row_count=None, details={'field': 'cat_infer_bad', 'values_size': 0, 'extendable': True, 'note': 'candidate_bootstrap'}),
 Issue(level='info', code='DOM.INFERENCE.APPLIED', message="Se infirió el dominio efectivo de 'cat_infer_ok' a partir de los valores observados (1000 únicos sobre 20000 filas no nulas); el campo se mantiene como categórico.", field='cat_infer_ok', source_field=None, row_count=None, details={'field': 'cat


Codes detectados: ['DOM.INFERENCE.APPLIED', 'DOM.INFERENCE.DEGRADED_TO_STRING', 'IMP.METADATA.DATASET_ID_CREATED', 'SCH.DOMAIN.EMPTY_VALUES']
Campos con SCH.DOMAIN.EMPTY_VALUES: ['cat_infer_bad', 'cat_infer_ok']
dtype_effective: {'movement_id': 'string', 'user_id': 'string', 'origin_time_utc': 'datetime', 'destination_time_utc': 'datetime', 'cat_infer_ok': 'categorical', 'cat_infer_bad': 'string', 'cat_defined': 'categorical'}
domains_effective keys: ['cat_defined', 'cat_infer_ok']
OK: en el caso de mayor tamaño se verificó que la política sigue comportándose igual al cambiar la escala del dataset.
